# Modelos de ensemble — escolha da biblioteca de boosting

**Card 04A, issue #185.** Este notebook responde a uma pergunta só: o projeto deve usar o `HistGradientBoostingClassifier`, que já vem no scikit-learn, ou o `XGBClassifier`, que acrescenta uma dependência ao projeto e à sessão do Colab?

A pergunta não é de gosto. Os cards #186, #187 e #188 rodam uma busca aleatória de 40 iterações sobre os folds do contrato, e uma sessão do Colab não dura para sempre. Se um ajuste único custar tempo demais, a busca não cabe na sessão e os três cards seguintes travam. O que decide, então, é medição: quanto custa **um** ajuste completo de cada biblioteca sobre a matriz que o contrato entrega.

**O que este notebook não faz.** Ele não treina o modelo final, não roda busca de hiperparâmetros e não cria nenhuma partição ou validação cruzada própria. As partições são as do contrato do card 01 (`matriz.preparar_matriz`), e nenhuma outra: uma fatia aleatória criada aqui contrariaria o agrupamento por Cliente da seção 3 e ainda mediria um treino menor do que o real, devolvendo um tempo otimista.

## 1. Ambiente e versões

As três bibliotecas que entram na comparação estão fixadas em versão exata no `requirements.txt` desde o commit 1 deste card. O motivo é a própria medição: comparar tempo entre duas bibliotecas só tem sentido se a sessão do Colab e a máquina de quem revisa instalarem a mesma pilha. Com faixa (`>=`), duas execuções do mesmo notebook em dias diferentes podem medir versões diferentes e a diferença de tempo passaria a ser da versão, não da biblioteca.

A célula abaixo instala o `requirements.txt` e imprime as versões efetivamente carregadas.

In [ ]:
!pip install -q -r requirements.txt

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost

# Inclui as pastas de código tanto no projeto local quanto no Colab. O break
# evita que duas cópias do mesmo módulo fiquem no caminho de importação.
for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break
for pasta_scripts in [Path("scripts"), Path("../scripts"), Path("/content/scripts")]:
    if pasta_scripts.exists():
        sys.path.insert(0, str(pasta_scripts.resolve()))
        break

import ensembles
importlib.reload(ensembles)

# Falha aqui, e nao tres secoes depois, se a sessao carregou uma versao
# diferente da fixada no requirements.txt (revisao da !94).
versoes_conferidas = ensembles.conferir_versoes()

print("scikit-learn", sklearn.__version__)
print("scipy       ", scipy.__version__)
print("xgboost     ", xgboost.__version__)
print("python      ", sys.version.split()[0])
print()
print("conferidas contra o requirements.txt:", versoes_conferidas)

O output acima é a evidência do CR01: as três versões impressas são exatamente as declaradas no `requirements.txt` (`scikit-learn==1.9.1`, `scipy==1.18.1`, `xgboost==3.4.1`), e a última linha mostra o que `ensembles.conferir_versoes` validou. A célula não depende de quem lê reparar numa divergência: se a sessão do Colab carregou uma cópia pré-instalada na frente do arquivo, `conferir_versoes` levanta `AssertionError` com o pacote, a versão esperada e a carregada, e o notebook para aqui. Nesse caso, reinicie o ambiente de execução depois do `pip install` e rode de novo.

A versão do scikit-learn passou de 1.9.0 para 1.9.1 ao incorporar a `develop`, que já fixava a 1.9.1 pelo #206; duas versões exatas da mesma biblioteca no projeto quebrariam a reprodutibilidade que as duas issues pedem.

## 2. Contrato de entrada (card 01)

Tudo o que a medição consome vem de `matriz.preparar_matriz`: as features (`x`), o alvo (`y`), o agrupamento por Cliente (`grupos`), as matrizes já transformadas e o `ColumnTransformer` ajustado **somente no treino**. Nada é reprocessado aqui.

Isso não é formalidade. O pré-processador do contrato imputa com mediana e indicador de ausência, escalona de forma robusta e codifica as categóricas em one-hot; um pré-processamento improvisado neste notebook produziria menos colunas e um ajuste mais rápido do que o que a busca do card #187 vai enfrentar de verdade. O número medido seria otimista justamente na direção que faria a decisão sair errada.

In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

import matriz
importlib.reload(matriz)
from matriz import preparar_matriz, resumo_da_matriz

CAMINHOS_POSSIVEIS = [
    Path("data/processed/base_analitica.parquet"),
    Path("../data/processed/base_analitica.parquet"),
    Path("/content/data/processed/base_analitica.parquet"),
    Path("/content/drive/MyDrive/data/processed/base_analitica.parquet"),
]
caminho = next((c for c in CAMINHOS_POSSIVEIS if c.exists()), None)
if caminho is None:
    raise FileNotFoundError(
        "Base analítica não encontrada. Rode o pre-processamento.ipynb antes, ou "
        f"coloque o parquet em um destes caminhos: {[str(c) for c in CAMINHOS_POSSIVEIS]}"
    )

df = pd.read_parquet(caminho)

# Datas do registro de decisão da política de particionamento (#127/#128), as
# mesmas do modelagem.ipynb. Elas são parâmetro, nunca constante de módulo.
CORTE_VALIDACAO = "2025-06-01"
CORTE_TESTE = "2025-12-01"

preparo = preparar_matriz(df, CORTE_VALIDACAO, CORTE_TESTE)
matriz_treino = preparo["matrizes"]["treino"]
y_treino = preparo["y"]["treino"]

print(resumo_da_matriz(preparo))
print()
print("colunas da matriz:", preparo["metadados"]["colunas_da_matriz"])
print("Clientes distintos no treino:", preparo["grupos"]["treino"].nunique())

O output acima mostra as três partições com n, intervalo de datas e Clientes. Dois números importam para ler a medição da seção 3: o **n do treino**, porque é sobre ele que o ajuste acontece, e o **número de colunas da matriz**, porque num boosting por histograma o custo depende do número de colunas e de bins muito mais do que do número de linhas.

Repare que o treino é bem menor que a base inteira: o corte é temporal e por Cliente, então o treino é só o período mais antigo. Estimar o custo da busca a partir do total de registros da base superestimaria o tempo; o número que vale é o do treino.

## 3. Medição de custo de um ajuste

`ensembles.medir_bibliotecas` ajusta cada biblioteca **uma** vez sobre a matriz de treino do contrato e devolve tempo de ajuste, pico de memória do processo e a semente usada.

Três cuidados estão embutidos nessa função e são o que torna a comparação honesta:

1. **Hiperparâmetros equivalentes termo a termo.** Mesmo passo, mesmo número de árvores, mesmo número de folhas, mesmo número de bins, histograma nos dois lados. A célula abaixo usa `ensembles.HIPERPARAMETROS_COMPARACAO`, que **não é uma cópia**: passo, número de árvores e número de folhas são lidos de `modelo.HIPERPARAMETROS_CANDIDATO`, os hiperparâmetros do candidato que o projeto realmente treina. A célula imprime os dois lado a lado e falha se divergirem, e o teste `test_comparacao_usa_os_hiperparametros_do_candidato` trava a mesma igualdade fora do notebook.
2. **Sem parada antecipada nos dois lados.** Com ela ligada, o tempo medido seria o de menos árvores do que as declaradas — e, no caso do scikit-learn, a fatia de parada seria aleatória, ignorando o agrupamento por Cliente.
3. **Estimador clonado a cada medição.** Um modelo reaproveitado começaria de um ajuste pronto e pareceria mais barato.

O pico de memória sai do RSS do processo, amostrado em thread separada. Quase toda a alocação das duas bibliotecas é nativa, e `tracemalloc` enxerga só o lado Python: mediria alguns megabytes e diria que as duas custam o mesmo.

In [ ]:
import ensembles
importlib.reload(ensembles)
from ensembles import escolher_biblioteca, estimar_busca, medir_bibliotecas

SEMENTE = ensembles.SEMENTE_PADRAO

import modelo

candidato = modelo.HIPERPARAMETROS_CANDIDATO
print("hiperparâmetros da comparação:", ensembles.HIPERPARAMETROS_COMPARACAO)
print("hiperparâmetros do candidato: ",
      {k: candidato[k] for k in ("learning_rate", "max_iter", "max_leaf_nodes")})

# O tempo medido so e o do modelo treinado se os tres valores forem os mesmos.
assert ensembles.HIPERPARAMETROS_COMPARACAO["passo"] == candidato["learning_rate"]
assert ensembles.HIPERPARAMETROS_COMPARACAO["arvores"] == candidato["max_iter"]
assert ensembles.HIPERPARAMETROS_COMPARACAO["folhas"] == candidato["max_leaf_nodes"]

tabela_medicao = medir_bibliotecas(matriz_treino, y_treino, random_state=SEMENTE)
tabela_medicao[["tempo_ajuste_s", "pico_memoria_mb", "delta_memoria_mb", "random_state"]]

O output acima é a tabela do CR04. Ela tem uma linha por biblioteca, com o tempo de um ajuste completo em segundos, o pico de memória do processo em MB, o quanto desse pico foi acrescentado pelo próprio ajuste (`delta_memoria_mb`) e o `random_state` usado.

O `delta` está ali porque o pico sozinho engana: ele inclui a base carregada e a matriz transformada, que já estavam na memória antes do ajuste e são as mesmas para as duas bibliotecas. Quem diferencia uma da outra é o delta.

### 3.1. Medição registrada

Execução de 17/09/2026, semente 42, scikit-learn 1.9.0 (antes do alinhamento à 1.9.1 descrito na seção 1), sobre uma matriz de treino de 167.815 linhas e 31 colunas:

| Biblioteca | Tempo de um ajuste (s) | Pico de memória (MB) | Acréscimo do ajuste (MB) | `random_state` |
| --- | ---: | ---: | ---: | ---: |
| `HistGradientBoostingClassifier` | 3,58 | 582,4 | 33,7 | 42 |
| `XGBClassifier` | 2,65 | 567,4 | 22,5 | 42 |

**Ressalva sobre esta execução, que precisa ser lida antes de usar os números.** Não havia base da Azul na máquina onde a tabela acima foi medida, e o `.gitignore` do projeto proíbe versioná-la. Os números vieram de uma base **sintética com a mesma forma declarada na issue** — 484.915 registros e 407.139 valores distintos de `ID_GOLDENRECORD`, passando pelo mesmo `preparar_matriz` e produzindo as mesmas 31 colunas de matriz. O que isso mede bem é a ordem de grandeza e a razão entre as duas bibliotecas; o que isso não mede é o efeito da distribuição real das categorias sobre o número de bins efetivos.

Além disso, a medição rodou em máquina local com 14 CPUs lógicas e 15,5 GB de RAM. Uma sessão gratuita do Colab tem cerca de 2 vCPUs, e as duas bibliotecas paralelizam o ajuste: espere algo entre 5 e 7 vezes mais lento lá.

**Portanto:** ao rodar este notebook no Colab sobre a base real, substitua a tabela acima pelo output da célula da seção 3 e refaça a conta da seção 4. A decisão da seção 5 continua válida enquanto a conclusão da seção 4 continuar valendo — e a seção 4 diz explicitamente qual é o limite.

## 4. A busca dos cards seguintes cabe na sessão?

`estimar_busca` multiplica o tempo de um ajuste pelas 40 iterações da busca e pelo número de folds do contrato. A conta é deliberadamente otimista: conta só os ajustes e ignora transformação, cálculo de métrica e o fato de cada fold custar quase o treino inteiro. Serve como piso — se nem esse piso couber na sessão, a busca com certeza não cabe.

O orçamento de 3 horas é o que se assume disponível numa sessão de Colab antes de ela cair ou ser reciclada.

In [ ]:
ORCAMENTO_HORAS = 3.0
N_ITERACOES = 40
N_FOLDS = 5

for biblioteca, linha in tabela_medicao.iterrows():
    horas = estimar_busca(linha["tempo_ajuste_s"], N_ITERACOES, N_FOLDS)
    print(f"{biblioteca:32s} {horas:6.2f} h  ({N_ITERACOES} iterações x {N_FOLDS} folds)")

decisao = escolher_biblioteca(
    tabela_medicao,
    orcamento_horas=ORCAMENTO_HORAS,
    n_iteracoes=N_ITERACOES,
    n_folds=N_FOLDS,
)
print()
print("escolhida:", decisao["escolhida"])
print(decisao["justificativa"])

O output acima aplica o critério do CR05, e a ordem dele importa: primeiro descarta quem não cabe no orçamento da sessão, porque uma biblioteca que estoura a sessão não é uma opção mais lenta, é uma opção que não existe; entre as que cabem, vence a mais barata de manter.

Na execução registrada, as duas cabem com folga — 0,20 h para o `HistGradientBoostingClassifier` e 0,15 h para o `XGBClassifier`, contra um orçamento de 3 h. Mesmo aplicando o fator de 5 a 7 vezes do Colab, as duas seguem abaixo de 1,5 h. **O limite onde essa conclusão mudaria:** um ajuste precisaria custar mais de 54 segundos no ambiente onde a busca for rodar (3 h ÷ 200 ajustes). Se o output da sua execução passar disso, a decisão da seção 5 precisa ser refeita.

## 5. Decisão registrada

**O projeto usa o `HistGradientBoostingClassifier` como biblioteca de boosting dos modelos de ensemble.**

A justificativa tem duas partes, nesta ordem:

1. **Viabilidade.** As duas bibliotecas cabem no orçamento da sessão com margem larga. A busca de 40 iterações vezes 5 folds sai em 0,20 h com o `HistGradientBoostingClassifier` e 0,15 h com o `XGBClassifier` na medição registrada; mesmo no pior fator esperado do Colab, nenhuma das duas se aproxima das 3 h. Ou seja, o custo **não** é critério de desempate aqui — a diferença de 0,93 s por ajuste não muda o que é possível fazer.
2. **Custo de manutenção.** Como a viabilidade não separa as duas, decide o que sobra: o `HistGradientBoostingClassifier` já está no scikit-learn, que o projeto usa desde a seção 4.2.2, e não acrescenta dependência ao `requirements.txt` nem instalação à sessão do Colab. O `XGBClassifier` acrescenta as duas coisas em troca de 26% de tempo num ajuste que já custa segundos.

**O que faria esta decisão ser revista.** O `XGBClassifier` foi medido mais rápido, e a vantagem dele cresce com o tamanho do problema. Se um card futuro aumentar a matriz — mais features, mais categorias, mais bins — a ponto de um ajuste passar dos 54 segundos no ambiente de execução, a ordem do critério se inverte e a dependência extra passa a se pagar. Por isso o `xgboost` fica declarado no `requirements.txt` mesmo não sendo o escolhido agora: a medição precisa continuar reproduzível para quem refizer essa conta.

**Comunicação.** Esta decisão vale para os cards #186, #187 e #188 e precisa chegar à dupla de Métricas e Decisões, que depende dela para dimensionar a busca.

## 7. Linha de base do Random Forest (card 08A, #187)

**Card 08A, issue #187.** Antes de a busca do #189 variar qualquer hiperparâmetro, o Random Forest precisa de um ponto de partida medido: sem ele, o ganho que a busca reportar não tem contra o que ser lido, e qualquer número que ela produzir parece bom.

O modelo não entra solto. `ensembles.criar_pipeline_random_forest` encadeia num único `Pipeline` o `ColumnTransformer` do contrato, `preparo["preprocessador"]` da seção 2, e um `RandomForestClassifier`. O pré-processador entra **clonado**: mesma especificação do contrato, sem o ajuste que `preparar_matriz` já fez no treino. É isso que permite à busca do #189 reajustar imputação e escala dentro de cada fold, em vez de herdar medianas que já viram as linhas de validação.

Duas decisões desta seção, declaradas aqui para não ficarem implícitas:

1. **Hiperparâmetros no padrão da biblioteca.** A linha de base não escolhe profundidade, número de árvores nem `class_weight`; quem escolhe é a busca.
2. **`random_state` = `SEMENTE` (42), a mesma da seção 3.** O Random Forest sorteia a amostra de cada árvore e as colunas de cada divisão; sem semente fixa, dois ajustes idênticos dariam florestas diferentes e a diferença de métrica na busca seria indistinguível de sorte.

A célula abaixo monta o pipeline e imprime os passos e os parâmetros do estimador, sem ajustar nada ainda.

In [ ]:
from ensembles import PASSO_MODELO, PASSO_PREPARO, criar_pipeline_random_forest

pipeline_rf = criar_pipeline_random_forest(preparo["preprocessador"], random_state=SEMENTE)

print("passos:", [nome for nome, _ in pipeline_rf.steps])
print("pré-processador é o objeto do contrato?", pipeline_rf.named_steps[PASSO_PREPARO] is preparo["preprocessador"])
print()
parametros_rf = pipeline_rf.named_steps[PASSO_MODELO].get_params()
for chave in ("n_estimators", "max_depth", "min_samples_leaf", "max_features", "class_weight", "random_state"):
    print(f"{chave:18s} {parametros_rf[chave]}")

O output acima tem três leituras. Os passos precisam ser `['preparo', 'modelo']`, os mesmos nomes do pipeline da logística (#208), porque é por `passo__parametro` que as grades das duplas são endereçadas. A pergunta sobre o objeto do contrato precisa responder `False`: o pipeline trabalha sobre uma cópia virgem, e o `preparo["preprocessador"]` usado nas seções anteriores continua intacto. E os parâmetros impressos precisam ser os padrões do scikit-learn (100 árvores, profundidade livre, `max_features='sqrt'`, sem `class_weight`), com `random_state` igual a 42.

### 7.1. Medição da linha de base

`ensembles.medir_linha_de_base` ajusta o pipeline sobre a partição de treino **crua**, `preparo["x"]["treino"]`, pontua a partição de validação e entrega `y_true`, `y_pred` e `y_proba` à função `avaliar` do card 05 (#241). Nenhuma métrica é calculada aqui e nenhum particionador é instanciado: o número que sai é o que `avaliar` devolver, com os mesmos nomes de métrica que as outras duplas vão reportar.

A matriz crua é deliberada. Passar `preparo["matrizes"]["treino"]`, que já saiu do pré-processador, faria o `ColumnTransformer` do pipeline ser ajustado sobre a saída de outro, sem erro nenhum e com colunas que a base não tem.

**Enquanto o #241 não estiver em `develop`, a célula abaixo não mede nada** e diz isso no output. É o caminho que a própria issue #187 prevê para esse atraso, e ela proíbe explicitamente contorná-lo com validação cruzada própria. Quando o #241 chegar, basta ajustar `MODULO_AVALIAR` ao nome do módulo publicado e reexecutar.

In [ ]:
from ensembles import medir_linha_de_base

MODULO_AVALIAR = "avaliacao"  # ajustar ao módulo que o card 05 (#241) publicar em src/

try:
    avaliar = getattr(importlib.import_module(MODULO_AVALIAR), "avaliar")
except (ImportError, AttributeError):
    avaliar = None

if avaliar is None:
    linha_de_base_rf = None
    print(f"avaliar() ainda não está disponível em src/{MODULO_AVALIAR}.py (card 05, #241).")
    print("Linha de base não medida; nenhuma métrica substituta foi calculada.")
else:
    linha_de_base_rf = medir_linha_de_base(
        pipeline_rf,
        preparo["x"]["treino"],
        preparo["y"]["treino"],
        preparo["x"]["validacao"],
        preparo["y"]["validacao"],
        avaliar=avaliar,
    )
    print("random_state:", linha_de_base_rf["random_state"])
    print(f"treino: {linha_de_base_rf['n_treino']} linhas | validação: {linha_de_base_rf['n_avaliacao']} linhas")
    print(f"tempo total (ajuste + pontuação): {linha_de_base_rf['tempo_total_s']:.1f} s")
    print()
    for nome, valor in linha_de_base_rf["metricas"].items():
        print(f"{nome:28s} {valor:.4f}")

O output acima tem uma de duas formas. Se `avaliar` ainda não existe, ele diz que a linha de base não foi medida, e é isso que deve ficar registrado: um número calculado por métrica própria entraria na tabela comparativa do card 18A.1 sem ser comparável com os das outras duplas. Se `avaliar` existe, ele imprime a semente, o tamanho das duas partições, o tempo total de ajuste e pontuação e as métricas exatamente com os nomes que `avaliar` devolveu (F2, Sensibilidade, Precisão Média e ROC-AUC, pelo #241).

O tempo total importa para o #189 do mesmo jeito que o da seção 3 importou para a escolha da biblioteca: multiplicado pelas iterações e pelos folds da busca, ele diz se a busca do Random Forest cabe na sessão do Colab.

### 7.2. Linha de base registrada

| Campo | Valor |
| --- | --- |
| `random_state` | 42 |
| Hiperparâmetros | padrão do scikit-learn 1.9.1, a versão fixada no `requirements.txt` |
| Partição medida | validação do contrato (de `2025-06-01` a `2025-12-01`) |
| Folds | nenhum: `avaliar(y_true, y_pred, y_proba)` do #241 recebe vetores e não particiona |
| Métricas de `avaliar` | **pendente do #241** |
| Tempo total | **pendente da execução no Colab sobre a base real** |

**Ressalva.** A issue #187 descreve `avaliar(modelo, X, y, groups)` e pede o número de folds informado por ela. O card dono da função, o #241, define outra assinatura, `avaliar(y_true, y_pred, y_proba)`, que não recebe modelo nem particiona; o pipeline segue a do #241, a mesma que o pipeline da logística (#208) já usa. Por isso a linha de base é medida na partição de validação, e a medida por fold fica para a busca do #189, que usa os folds do contrato.

## 6. Dependências e próximos passos

**Depende de:** card 01 (contrato de dados, Kaylan e Pedro) — `x`, `y`, `grupos` e o `ColumnTransformer`, consumidos na seção 2 via `matriz.preparar_matriz`.

**Bloqueia:** #186, #187 e #188, que só podem dimensionar a busca depois da decisão da seção 5. A seção 7 (#187) depende ainda do card 05 (#241), dono de `avaliar`, e bloqueia a busca do Random Forest (#189).

**Próximo passo obrigatório:** rodar este notebook em sessão nova do Colab, sobre a base analítica real, e substituir a tabela da seção 3.1 e os campos pendentes da seção 7.2 pelos outputs medidos lá. As travas que protegem a validade da medição estão em `tests/test_ensembles.py` e rodam com `pytest tests/test_ensembles.py -v`.